# Data managment for the GNN

Generating the data matrices with the oservables and their labels for the GNN training later on!

In [1]:
import numpy as np
from Stellar_objects.cluster_v4_properties import *
from Stellar_objects.visu import *
from Stellar_objects.dispersion_vel_LOS import *
from plot_swift_output import load_snapshot

from bound_labels import compute_bound_stars

from adri_plotstyle import init_plot_style
init_plot_style()
import numpy as np
import matplotlib.pyplot as plt

✅ Matplotlib LaTeX style initialisé (Police: helvet, Taille: 14)
✅ Matplotlib LaTeX style initialisé (Police: helvet, Taille: 12)


In [12]:
import os
folder = "outputs_SWIFT_jed/withoutDF/"
outputs_folders = [f for f in os.listdir(folder) if f.startswith("IC_28")]
for path in outputs_folders:
    print(f"Processing folder: {path}")
    path = folder+path
    outputs_list = [f for f in os.listdir(path) if f.startswith("snapshot_")]
    #output_list = ["snapshot_0080.hdf5"]  # Example: specify the exact snapshot file
    labels_folder = "GNN-jed/labels_bound_stars/"
    for output in outputs_list:
        fname = os.path.join(path, output)
        pos, vel, _, _ = load_snapshot(fname, shift=500.0)
        is_bound, _, _, _, _, _, _, _, _, _ = compute_bound_stars(pos[:,0], pos[:,1], pos[:,2], vel[:,0], vel[:,1], vel[:,2], np.ones_like(pos[:,0]))

        IC_labels = path.split("/")[-1]  # Extracting the IC name from the path
        location  = labels_folder + IC_labels
        os.makedirs(location, exist_ok=True)

        #saving the bound labels to a .npy file
        np.save(f"{location}/B_L_{output.split('.')[0]}.npy", is_bound)

Processing folder: IC_28
/Users/adriancozma/Desktop/Ecole/MA2/TP4b/simulations-SWIFT/bound_labels.py:119: RuntimeWarning: invalid value encountered in scalar divide
  e_crit = - (3 * G * m_cluster) / (2 * rj)



In [9]:
# # Example: Load the saved labels
is_bound = np.load(f"{location}/B_L_{output.split('.')[0]}.npy")
print(is_bound)

[False False False ... False False False]


### Loading raw data, projecting and rotating it and normalizing it for the GNN training dataset  

In [2]:
from observables_matrix import *

In [6]:
import os
folder = "outputs_SWIFT_jed/withoutDF/"
outputs_folders = [f for f in os.listdir(folder) if f.startswith("IC_")]
for path in outputs_folders:
    print(f"Processing folder: {path}")
    path = folder+path
    #save_folder = "Simulations_jed/IC_1_bis/figures/"
    output_list = [f for f in os.listdir(path) if f.startswith("snapshot_")]
    #outputs_list = ["snapshot_0080.hdf5"]  # Example: specify the exact snapshot file

    folder_obs = "GNN-jed/obs_all_stars/"

    for output in output_list:
        fname = os.path.join(path, output)
        pos_raw, vel_raw, _, _ = load_snapshot(fname, shift=500.0)

        final_matrices = [] # matix instance for each output fith all rotations 
                            # (shape (n, N, 6) where n is the number of rotations and N the number of stars)
        n = 5               # specify the number of rotations   
        for _ in range(n): # make n different rotations for augmenting the data
            pos_rot, vel_rot = rotate_data(pos_raw, vel_raw)
            matrix_obs = get_observables_matrix(pos_rot, vel_rot)

            #simple quick plot to check the rotations are working as expected (should be different for each rotation)
            if output == output_list[0]:
                plot_observables(matrix_obs, f"obs_output_{output}_rotation_{n}.png")


            X_ready, mu, sigma = standardise_matrix(matrix_obs)
            final_matrices.append(X_ready)
        IC_labels = path.split("/")[-1]  # Extracting the IC name from the path
        location = folder_obs + IC_labels

        os.makedirs(location, exist_ok=True)
        # saving the final matrices for each output and rotation
        final_matrices = np.array(final_matrices) # shape (n, N, 6)
        np.save(f"{location}/obs_matrices_{output.split('.')[0]}", final_matrices)
                
        # trying to save some memory
        del final_matrices, pos_raw, vel_raw, pos_rot, vel_rot, matrix_obs, X_ready, mu, sigma

Processing folder: IC_28
Processing folder: IC_17
Processing folder: IC_9
Processing folder: IC_19
Processing folder: IC_26
Processing folder: IC_21
Processing folder: IC_81
Processing folder: IC_75
Processing folder: IC_72
Processing folder: IC_86
Processing folder: IC_44
Processing folder: IC_43
Processing folder: IC_20
Processing folder: IC_6
Processing folder: IC_18
Processing folder: IC_1
Processing folder: IC_8
Processing folder: IC_11
Processing folder: IC_29
Processing folder: IC_16
Processing folder: IC_42
Processing folder: IC_87
Processing folder: IC_73
Processing folder: IC_74
Processing folder: IC_80
Processing folder: IC_69
Processing folder: IC_51
Processing folder: IC_60
Processing folder: IC_34
Processing folder: IC_33
Processing folder: IC_61
Processing folder: IC_59
Processing folder: IC_66
Processing folder: IC_50
Processing folder: IC_68
Processing folder: IC_32
Processing folder: IC_35
Processing folder: IC_71
Processing folder: IC_82
Processing folder: IC_76
Proc